In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time

In [2]:
# Ensure we are running on GPU with appropriate precision

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

In [19]:
class BenchmarkGQAEngine(nn.Module):
    def __init__(self, d_model=512, n_heads=8, n_groups=2):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_groups = n_groups
        self.head_dim = d_model // n_heads
        self.heads_per_group = n_heads // n_groups

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, n_groups * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_groups * self.head_dim)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, past_cache=None, use_flash=False):
        B, S, D = x.shape
        
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_groups, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_groups, self.head_dim).transpose(1, 2)

        if past_cache is not None:
            past_k, past_v = past_cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)
        current_cache = (k, v)

        if use_flash and device == "cuda":
            k_expanded = k.repeat_interleave(self.heads_per_group, dim=1)
            v_expanded = v.repeat_interleave(self.heads_per_group, dim=1)
            with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_math=False, enable_mem_efficient=False):
                out = F.scaled_dot_product_attention(q, k_expanded, v_expanded, is_causal=(S > 1))
        else:
            q_grouped = q.view(B, self.n_groups, self.heads_per_group, S, self.head_dim)
            k_grouped = k.unsqueeze(2)
            v_grouped = v.unsqueeze(2)
            scores = torch.matmul(q_grouped, k_grouped.transpose(-2, -1)) / (self.head_dim ** 0.5)
            if S > 1:
                mask = torch.tril(torch.ones(S, S, device=x.device)).view(1, 1, 1, S, S)
                scores = scores.masked_fill(mask == 0, float('-inf'))
            attn = torch.softmax(scores, dim=-1)
            out = torch.matmul(attn, v_grouped).view(B, self.n_heads, S, self.head_dim)

        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.out_proj(out), current_cache


In [5]:
# ------------------------------------------------------------------
# Benchmarking Engine Generation Loop Simulation
# ------------------------------------------------------------------

In [20]:
def run_generation_benchmark(model, prompt_len=512, gen_tokens=100, use_cache=True, use_flash=False):
    x_input = torch.randn(1, prompt_len, model.d_model, device=device, dtype=dtype)
    cache = None
    
    _, _ = model(x_input[:, :1, :], use_flash=use_flash)
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()
    out, cache = model(x_input, past_cache=cache, use_flash=use_flash)
    
    for i in range(gen_tokens):
        if use_cache:
            next_step_input = torch.randn(1, 1, model.d_model, device=device, dtype=dtype)
            out, cache = model(next_step_input, past_cache=cache, use_flash=use_flash)
        else:
            full_context_len = prompt_len + i + 1
            full_input = torch.randn(1, full_context_len, model.d_model, device=device, dtype=dtype)
            out, _ = model(full_input, past_cache=None, use_flash=use_flash)

    if device == "cuda":
        torch.cuda.synchronize()
    end_time = time.perf_counter()
    
    total_time = end_time - start_time
    tokens_per_sec = gen_tokens / total_time
    peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if device == "cuda" else 0.0
    
    return tokens_per_sec, peak_mem

In [21]:
model = BenchmarkGQAEngine().to(device).to(dtype)

print("--- BENCHMARK RESULTS ---")
tps_base, mem_base = run_generation_benchmark(model, use_cache=False, use_flash=False)
print(f"Standard Loop (No Cache):   {tps_base:.2f} tokens/sec | Peak Memory: {mem_base:.2f} MB")

tps_opt1, mem_opt1 = run_generation_benchmark(model, use_cache=True, use_flash=False)
print(f"Optimized (GQA + Cache):     {tps_opt1:.2f} tokens/sec | Peak Memory: {mem_opt1:.2f} MB")

tps_opt2, mem_opt2 = run_generation_benchmark(model, use_cache=True, use_flash=True)
print(f"Full Hardware Stack (Flash): {tps_opt2:.2f} tokens/sec | Peak Memory: {mem_opt2:.2f} MB")

--- BENCHMARK RESULTS ---
Standard Loop (No Cache):   366.44 tokens/sec | Peak Memory: 45.53 MB
Optimized (GQA + Cache):     898.18 tokens/sec | Peak Memory: 29.39 MB
Full Hardware Stack (Flash): 839.04 tokens/sec | Peak Memory: 21.92 MB
